# lora 训练 whisper-v3 使用 common-voice-27 地方语言训练

| 地区 | 训练说话人 | 训练条数 | 验证说话人 | 测试说话人/条数 |
|---|---:|---:|---:|---:|
| 北京 | 17 | 1,851 | 16 | 34 / 123 |
| 江苏 | 15 | 1,471 | 20 | 25 / 119 |
| 黑龙江 | 12 | 1,588 | 6 | 15 / 43 |
| 河南 | 11 | 989 | 11 | 21 / 86 |
| 浙江 | 11 | 1,292 | 21 | 21 / 102 |
| 四川 | 11 | 1,228 | 16 | 20 / 70 |
| 山东 | 10 | 1,035 | 34 | 26 / 125 |

## 缓存与路径

In [1]:
from pathlib import Path
import os
import re

# Hugging Face 缓存放到外接盘，避免占用 C 盘
HF_ROOT = "/sigan/ai-ml/huggingface"
os.environ["HF_HOME"] = HF_ROOT
os.environ["HF_HUB_CACHE"] = f"{HF_ROOT}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_ROOT}/datasets"

# Common Voice 27 中文目录
CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)
CLIPS_DIR = CV_ROOT / "clips"

# CV_ROOT = Path(
#     "/sigan/ai-ml/datasets/common-voice-27/chinese/"
#     "common-voice-scripted-speech-27-0-chines-b098f12c.partial/"
#     "cv-corpus-27.0-2026-09-11/zh-CN"
# )
# CLIPS_DIR = CV_ROOT / "clips"

# 如果解压完成后目录去掉了 .partial，把上面的路径改成实际路径
print(CV_ROOT)
print(CLIPS_DIR)

/home/sigan/datasets/common_voice_27_zh/cv-corpus-27.0-2026-09-11/zh-CN
/home/sigan/datasets/common_voice_27_zh/cv-corpus-27.0-2026-09-11/zh-CN/clips


## 加载 TSV 和音频

In [2]:
from datasets import DatasetDict, Audio, load_dataset

# 地区筛选，挑出相关地区的语音来
ACCENTS = {
    "出生地：11 北京市",
    "出生地：23 黑龙江省",
    "出生地：37 山东省",
    # "出生地：32 江苏省",
    # "出生地：41 河南省",
    # "出生地：33 浙江省",
    # "出生地：51 四川省",
}

# 读取一个 split：TSV 提供文本和文件名，clips 提供真实音频
def load_split(tsv_name):
    ds = load_dataset(
        "csv",
        data_files=str(CV_ROOT / tsv_name),
        delimiter="\t",
        keep_default_na=False,
    )["train"]

    ds = ds.filter(
        lambda row: row["accents"] in ACCENTS
    )

    # TSV 中只有文件名，这里补成能访问的绝对路径
    ds = ds.map(
        lambda batch: {
            "audio": [
                str(CLIPS_DIR / filename)
                for filename in batch["path"]
            ]
        },
        batched=True,
    )

    # 把字符串路径变成可解码的 16kHz 音频对象
    ds = ds.cast_column(
        "audio",
        Audio(sampling_rate=16000),
    )

    # 保留 client_id 和 accents 便于核查说话人与地区
    return ds.select_columns(
        ["audio", "sentence", "client_id", "accents"]
    )


common_voice = DatasetDict({
    "train": load_split("train.tsv"),
    "validation": load_split("dev.tsv"),
    "test": load_split("test.tsv"),
})

print(common_voice)
print(common_voice["train"][0])

Filter:   0%|          | 0/29814 [00:00<?, ? examples/s]

Map:   0%|          | 0/4474 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10679 [00:00<?, ? examples/s]

Map:   0%|          | 0/884 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10679 [00:00<?, ? examples/s]

Map:   0%|          | 0/291 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence', 'client_id', 'accents'],
        num_rows: 4474
    })
    validation: Dataset({
        features: ['audio', 'sentence', 'client_id', 'accents'],
        num_rows: 884
    })
    test: Dataset({
        features: ['audio', 'sentence', 'client_id', 'accents'],
        num_rows: 291
    })
})


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


{'audio': {'path': '/home/sigan/datasets/common_voice_27_zh/cv-corpus-27.0-2026-09-11/zh-CN/clips/common_voice_zh-CN_21602278.mp3', 'array': array([ 5.45696821e-12, -2.18278728e-11, -1.09139364e-11, ...,
       -1.06474454e-06, -9.89422460e-07, -1.88279614e-06]), 'sampling_rate': 16000}, 'sentence': '这种子流派的游戏剧情通常具有较强的暴力犯罪主题。', 'client_id': '55a7e303bbc54b407ccdc5789dc21cf745ab32adf4896036f0d189b5c3ae589d77e292ee95791b6919120dcd5ffd6f31e97b68a791b45a116f4c785d09f17b88', 'accents': '出生地：23 黑龙江省'}


## 加载 Processor 并预处理

In [3]:
from transformers import AutoProcessor

MODEL_ID = "openai/whisper-large-v3"

# large-v3 使用 128 个 Mel 频带，必须使用它自己的 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor


def prepare_dataset(batch):
    audio = batch["audio"]

    # 音频转 Log-Mel 特征，large-v3 输出约 [128, 3000]
    batch["input_features"] = (
        feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
        )
        .input_features[0]
    )

    # 正确文本转 token ID，作为训练 labels
    batch["labels"] = tokenizer(
        batch["sentence"]
    ).input_ids

    return batch


# 29,814 条全量训练数据都处理；外接盘较慢时把 num_proc 改成 2 或 1
tokenized_dataset = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice["train"].column_names,
    num_proc=4,
    writer_batch_size=100,
)

# 验证集有 10,679 条，每次生成评估太慢，固定抽 1,000 条
eval_dataset = (
    tokenized_dataset["validation"]
    .shuffle(seed=42)
    # .select(range(300)) # 可能地区筛选后不足该数字，取全量
)

# 用完整训练集训练，最后用完整测试集评估
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

print("train：", len(train_dataset))
print("validation subset：", len(eval_dataset))
print("test：", len(test_dataset))

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map (num_proc=4):   0%|          | 0/4474 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/884 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/291 [00:00<?, ? examples/s]

train： 4474
validation subset： 884
test： 291


## Data Collator

这组目标模块会同时覆盖 Encoder、Decoder、交叉注意力和 FFN。没有加入 proj_out，因为它是语言模型输出头，参数量大，通常不值得用 LoRA 训练。

In [4]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import jiwer
import torch

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


# 把不同长度的文本标签补齐，音频特征也用 Processor 对齐
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [
            {"input_features": x["input_features"]}
            for x in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": x["labels"]}
            for x in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        if (
            labels[:, 0]
            == self.processor.tokenizer.bos_token_id
        ).all().cpu().item():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels,
        }


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

## 定义指标

In [5]:
import re
import unicodedata

import opencc

# 繁体转简体
_t2s = opencc.OpenCC("t2s")

# 统一常见中英文标点
_PUNCT_MAP = str.maketrans({
    ",": "，",
    "?": "？",
    "!": "！",
    ":": "：",
    ";": "；",
    "(": "（",
    ")": "）",
    "[": "【",
    "]": "】",
    "{": "｛",
    "}": "｝",
    "、": "，",
})

# 简繁转换与标点归一化函数
def normalize_chinese(text, remove_punctuation=False):
    """
    Chinese ASR 文本统一：
    1. 繁体 -> 简体
    2. 全角字符标准化
    3. 英文转小写
    4. 删除空格
    5. 统一常见标点
    6. 可选删除全部标点
    """
    if text is None:
        return ""

    # 先统一繁简体
    text = _t2s.convert(text)

    # 将全角字母、数字等统一为半角形式
    text = unicodedata.normalize("NFKC", text)

    # 英文大小写统一
    text = text.lower()

    # 中文 ASR 通常不依赖空格
    text = re.sub(r"\s+", "", text)

    # 英文句点不误伤小数，例如 3.14 保持不变
    text = re.sub(r"(?<!\d)\.(?!\d)", "。", text)

    # 统一中英文标点
    text = text.translate(_PUNCT_MAP)

    # 连续标点压缩成一个
    text = re.sub(r"，+", "，", text)
    text = re.sub(r"。+", "。", text)
    text = re.sub(r"？+", "？", text)
    text = re.sub(r"！+", "！", text)

    # 如果只比较识别内容，可以删除全部标点
    if remove_punctuation:
        text = re.sub(
            r"[，。？！：；、（）【】｛｝,\.\?!:;()\[\]{}]",
            "",
            text,
        )

    return text.strip()

In [6]:
# 中文没有天然空格，CER 比 WER 更适合作为主要指标
def compute_metrics(result):
    pred_ids = result.predictions
    label_ids = result.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    raw_predictions = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True,
    )
    
    raw_references = tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True,
    )
    
    predictions = [
        normalize_chinese(
            text,
            remove_punctuation=True,
        )
        for text in raw_predictions
    ]
    
    references = [
        normalize_chinese(
            text,
            remove_punctuation=True,
        )
        for text in raw_references
    ]

    # 过滤掉归一化后为空的参考文本，避免 jiwer 报错
    valid_pairs = [
        (prediction, reference)
        for prediction, reference in zip(
            predictions,
            references,
        )
        if reference.strip()
    ]

    empty_reference_count = (
        len(references) - len(valid_pairs)
    )

    # 整个批次都没有有效参考文本时，不计算 CER
    if not valid_pairs:
        return {
            "cer": 1.0,
            "empty_references": float(empty_reference_count),
        }

    valid_predictions, valid_references = zip(
        *valid_pairs
    )

    return {
        "cer": float(
            jiwer.cer(
                reference=list(valid_references),
                hypothesis=list(valid_predictions),
            )
        ),
        "empty_references": float(empty_reference_count),
    }


# 生成 token 后再交给 compute_metrics
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

## 定义模型训练配置

In [7]:
# 8-bit 加载基座，显存更适合 RTX 4080
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 训练时关闭 KV Cache，并清除推理阶段的固定解码限制
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 训练过程中的生成式评估也固定使用中文转写
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="chinese",
    task="transcribe",
)
# 不加这一项，trainer.predict() 可能使用错误的语言任务标记
model.generation_config.forced_decoder_ids = (
    forced_decoder_ids
)

# 开启梯度检查点，进一步降低训练激活显存
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
# 全量线性模块：注意力 Q/K/V/O 加 FFN fc1/fc2
target_modules = [
    "q_proj",
    "k_proj",
    
    # 之前 v3 full 修改全部注意力、交叉注意力和 FFN；
    # 更新范围太大，导致 decoder 生成重复；
    # q/v 更新范围更小，更适合在基座已经较强时做小幅适配。
    # "v_proj",
    # "out_proj",
    # "fc1",
    # "fc2",
]

# 只训练 LoRA 参数，冻结量化后的 Whisper 基座
lora_config = LoraConfig(
    r=8,
    # lora_alpha=32,
    lora_alpha=8,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
)
# 2026-11-10 第一轮 epoch=1 跑完，先做短训练，adapter 已让 Whisper 的 decoder 退化
# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=8,          # 从 32 降到 8，缩放倍率变成 1
#     target_modules=[
#         "q_proj",
#         "v_proj",
#     ],
#     lora_dropout=0.05,
#     bias="none",
# )

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,932,160 || all params: 1,547,422,720 || trainable%: 0.2541102666503436


## 训练

In [9]:
OUTPUT_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-china-north-lora"
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # RTX 4080 16GB：物理 batch 2，累积 8 次，有效 batch=16
    per_device_train_batch_size=2,
    # 有效 batch = 2 × 8 = 16
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,

    # 先完整训练 1 轮
    num_train_epochs=1,
    # 当前 V3 已预训练功能强大，很容易推理退化，步数小点方便随时保存
    # max_steps=1000,
    
    # 低学习率，避免破坏 v3 原有生成能力
    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_torch",

    # RTX 4080 支持 FP16，混合精度训练应该开启
    fp16=True,
    bf16=False,

    # 每 1000 个优化步验证一次；日志不会自动保存模型
    # 每 250 步观察 CER 和保存 checkpoint
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,
    logging_steps=20,

    # 验证时真正生成文本，而不是只计算 teacher-forcing loss
    predict_with_generate=True,
    generation_max_length=225,

    # 用验证集 CER 选择效果最好的 checkpoint
    metric_for_best_model="cer",
    greater_is_better=False,
    load_best_model_at_end=True,

    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)
# 2026-11-10 第一轮 epoch=1 跑完，训练参数改为 300~500 冒烟测试
# training_args = Seq2SeqTrainingArguments(
#     output_dir="/sigan/ai-ml/models/whisper-large-v3-zh-smoke",

#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     per_device_eval_batch_size=2,

#     learning_rate=1e-5,    # 从 1e-4 降下来
#     max_steps=300,
#     warmup_ratio=0.1,
#     lr_scheduler_type="linear",

#     fp16=True,

#     evaluation_strategy="steps",
#     eval_steps=100,
#     save_strategy="steps",
#     save_steps=100,
#     save_total_limit=3,

#     predict_with_generate=True,
#     generation_max_length=225,

#     metric_for_best_model="cer",
#     greater_is_better=False,
#     load_best_model_at_end=True,

#     logging_steps=20,
#     remove_unused_columns=False,
#     label_names=["labels"],
#     report_to="none",
#     seed=42,
# )


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    tokenizer=processor.tokenizer,
)

In [10]:
# 如果已有 checkpoint，改为 trainer.train(resume_from_checkpoint=True)
trainer.train()
# trainer.train(resume_from_checkpoint=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss,Cer,Empty References
250,1.366100,1.279480,0.114206,0.000000


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=279, training_loss=1.3595779924837064, metrics={'train_runtime': 4148.2984, 'train_samples_per_second': 1.079, 'train_steps_per_second': 0.067, 'total_flos': 1.52068749262848e+19, 'train_loss': 1.3595779924837064, 'epoch': 1.0})

## 保存

In [11]:
# 保存 LoRA adapter 和 Processor；完整 Whisper 基座不需要重复保存
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("已保存：", OUTPUT_DIR)
# trainer.save_model("/sigan/ai-ml/models/whisper-large-v3-zh-smoke")
# processor.save_pretrained("/sigan/ai-ml/models/whisper-large-v3-zh-smoke")
# print("已保存：", "/sigan/ai-ml/models/whisper-large-v3-zh-smoke")

已保存： /sigan/ai-ml/models/whisper-large-v3-zh-china-north-lora


## 评估

In [12]:
# 验证集只用于调参，最终结果必须看没有参与训练的测试集
test_result = trainer.predict(
    test_dataset,
    metric_key_prefix="test",
    max_length=225,
    num_beams=1,
)

print(test_result.metrics)

{'test_loss': 1.2722655534744263, 'test_cer': 0.11075069508804448, 'test_empty_references': 0.0, 'test_runtime': 538.8538, 'test_samples_per_second': 0.54, 'test_steps_per_second': 0.271}


In [ ]:
# small_test = test_dataset.select(range(50))

# small_result = trainer.predict(
#     small_test,
#     metric_key_prefix="small_test",
#     max_length=225,
#     num_beams=1,
# )

# print(small_result.metrics)

# 测试音频

## 第一轮 北京方言 测试

In [ ]:
import csv
import gc
import shutil
import time
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from IPython.display import display
from peft import PeftModel
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
)

### 基础配置

In [ ]:
MODEL_ID = "openai/whisper-large-v3"

# 改成你训练完成后的 LoRA 目录
ADAPTER_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-beijing-lora"
)

CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 20 条测试音频同步到这个目录
TEST_AUDIO_DIR = Path(
    "/sigan/ai-ml/datasets/"
    "common-voice-27-beijing-test1"
)

# 测试时读取 source test.tsv
TEST_TSV = CV_ROOT / "test.tsv"

ACCENTS = {
    "出生地：11 北京市",
    # "出生地：23 黑龙江省",
    # "出生地：37 山东省",
    # "出生地：32 江苏省",
    # "出生地：41 河南省",
    # "出生地：33 浙江省",
    # "出生地：51 四川省",
}
NUM_SAMPLES = 20

SAMPLING_RATE = 16000
MAX_NEW_TOKENS = 100
NUM_BEAMS = 1
DEVICE = "cuda"

TEST_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

### 抽取处理数据

In [ ]:
# ============================================================
# 2. 中文文本归一化
# ============================================================

def clean_text(text):
    """标准答案和模型输出都使用完全相同的归一化规则。"""
    text = normalize_chinese(
        text,
        remove_punctuation=True,
    )

    # OpenCC t2s 不会把“妳”转换为“你”，这里单独统一
    return text.replace("妳", "你")


# ============================================================
# 3. 抽取 20 条北京测试音频并同步到目录
# ============================================================

with TEST_TSV.open(
    encoding="utf-8",
    newline="",
) as handle:
    reader = csv.DictReader(handle, delimiter="\t")

    test_rows = []
    selected_speakers = set()

    for row in reader:
        # 必须是 not in，不能再用 != TARGET_ACCENT
        if row["accents"] not in ACCENTS:
            continue

        if row["client_id"] in selected_speakers:
            continue

        test_rows.append(row)
        selected_speakers.add(row["client_id"])

        if len(test_rows) >= NUM_SAMPLES:
            break

if len(test_rows) != NUM_SAMPLES:
    raise RuntimeError(
        f"只找到 {len(test_rows)} 条目标地区测试数据，"
        f"不足 {NUM_SAMPLES} 条"
    )

# 同步音频：从原始 clips 复制到 test1 目录
manifest_rows = []

for index, row in enumerate(test_rows, start=1):
    filename = row["path"]
    source = CV_ROOT / "clips" / filename
    destination = TEST_AUDIO_DIR / filename

    # 复制音频；重复运行时覆盖成同一份文件
    shutil.copy2(source, destination)

    manifest_rows.append({
        "序号": index,
        "文件名": filename,
        "标准答案": row["sentence"],
        "地区": row["accents"],
        "说话人": row["client_id"][:8],
    })

manifest_df = pd.DataFrame(manifest_rows)

print("测试音频目录：", TEST_AUDIO_DIR)
print("已同步音频数量：", len(manifest_df))
display(manifest_df)

### 加载 Processor

In [ ]:
import re
import unicodedata
import opencc

_t2s = opencc.OpenCC("t2s")

def normalize_chinese(text, remove_punctuation=False):
    if text is None:
        return ""

    text = _t2s.convert(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"\s+", "", text)

    if remove_punctuation:
        text = re.sub(
            r"[，。？！：；、（）【】｛｝,\.\?!:;()\[\]{}]",
            "",
            text,
        )

    return text.strip()


def clean_text(text):
    text = normalize_chinese(
        text,
        remove_punctuation=True,
    )
    return text.replace("妳", "你")

In [ ]:
# ============================================================
# 4. 加载 Processor
# ============================================================

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="chinese",
    task="transcribe",
)


# ============================================================
# 5. 从同步后的目录读取音频并提取特征
# ============================================================

filenames = manifest_df["文件名"].tolist()
references = []
input_features = []

for index, row in enumerate(test_rows, start=1):
    # 注意：这里从 test1 目录读取，不再读取原始 clips
    audio_path = TEST_AUDIO_DIR / row["path"]

    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLING_RATE,
    )

    feature = processor(
        audio,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
    )["input_features"][0]

    # 标准答案也执行归一化
    references.append(
        clean_text(row["sentence"])
    )
    input_features.append(feature)

    print(f"已加载：{index}/{NUM_SAMPLES}")

### 生成数据

In [ ]:
def run_test(use_lora):
    """
    use_lora=False：原始 Whisper large-v3
    use_lora=True ：Whisper large-v3 + LoRA
    """

    print("\n加载模型，use_lora =", use_lora)

    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_8bit=True
        ),
        device_map="auto",
    )

    # 推理阶段开启 KV Cache，减少生成耗时
    base_model.config.use_cache = True

    model = base_model

    if use_lora:
        model = PeftModel.from_pretrained(
            base_model,
            ADAPTER_DIR,
        )

    model.eval()

    # 强制使用中文转写任务
    model.generation_config.forced_decoder_ids = (
        forced_decoder_ids
    )

    predictions = []
    start_time = time.perf_counter()

    for index, feature in enumerate(
        input_features,
        start=1,
    ):
        inputs = {
            "input_features": (
                feature
                .unsqueeze(0)
                .to(DEVICE)
                .half()
            )
        }

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )

        prediction = processor.batch_decode(
            output_ids,
            skip_special_tokens=True,
        )[0]

        # 模型输出也做归一化
        predictions.append(
            clean_text(prediction)
        )

        print(f"已生成：{index}/{NUM_SAMPLES}")

    elapsed = time.perf_counter() - start_time

    # 释放显存，准备加载下一个模型
    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, elapsed


# 原始 V3
base_predictions, base_time = run_test(
    use_lora=False
)

# V3 + 北京 LoRA
lora_predictions, lora_time = run_test(
    use_lora=True
)

### 计算统计

In [ ]:
# ============================================================
# 7. 计算总体 CER
# ============================================================

base_cer = jiwer.cer(
    reference=references,
    hypothesis=base_predictions,
)

lora_cer = jiwer.cer(
    reference=references,
    hypothesis=lora_predictions,
)

print("\n" + "=" * 50)
print("原始 V3 CER：", base_cer)
print("V3 LoRA CER：", lora_cer)
print("CER 降低量：", base_cer - lora_cer)
print("原始 V3 耗时：", round(base_time, 2), "秒")
print("LoRA 耗时：", round(lora_time, 2), "秒")
print("=" * 50)


# ============================================================
# 8. 生成逐条 results_df
# ============================================================

rows = []

for index, (
    filename,
    reference,
    base_prediction,
    lora_prediction,
) in enumerate(
    zip(
        filenames,
        references,
        base_predictions,
        lora_predictions,
    ),
    start=1,
):
    base_sample_cer = jiwer.cer(
        reference=reference,
        hypothesis=base_prediction,
    )

    lora_sample_cer = jiwer.cer(
        reference=reference,
        hypothesis=lora_prediction,
    )

    rows.append({
        "序号": index,
        "文件名": filename,
        "标准答案": reference,
        "原始V3输出": base_prediction,
        "LoRA输出": lora_prediction,
        "原始V3 CER": round(base_sample_cer, 4),
        "LoRA CER": round(lora_sample_cer, 4),
        # 正数表示 LoRA 更好
        "LoRA改善": round(
            base_sample_cer - lora_sample_cer,
            4,
        ),
    })

results_df = pd.DataFrame(rows)

display(results_df)

### 单独查看 LoRA 变差的样本

In [ ]:
worse_df = results_df[
    results_df["LoRA改善"] < 0
].sort_values(
    by="LoRA改善",
    ascending=True,
)

print("\nLoRA 更差的样本：", len(worse_df))
display(worse_df)

### 结果判断

原始 V3 CER： 0.10185185185185185
V3 LoRA CER： 0.10185185185185185
CER 降低量： 0.0
原始 V3 耗时： 44.8 秒
LoRA 耗时： 50.65 秒

1. 数据太少了，不足以支撑训练
2. Common Voice 的 accents 表示说话人自报或出生地区，不是“这段音频确实是北京话”的标签。文本也是标准书面中文朗读，所以最终没有形成方言训练效果

## 第二轮 北京+黑龙江+山东方言 测试

In [13]:
import csv
import gc
import shutil
import time
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from IPython.display import display
from peft import PeftModel
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
)

### 基础配置

In [14]:
MODEL_ID = "openai/whisper-large-v3"

# 改成你训练完成后的 LoRA 目录
ADAPTER_DIR = (
    "/sigan/ai-ml/models/whisper-large-v3-zh-china-north-lora"
)

CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_zh/"
    "cv-corpus-27.0-2026-09-11/zh-CN"
)

# 20 条测试音频同步到这个目录
TEST_AUDIO_DIR = Path(
    "/sigan/ai-ml/datasets/"
    "common-voice-27-beijing-test2"
)

# 测试时读取 source test.tsv
TEST_TSV = CV_ROOT / "test.tsv"

ACCENTS = {
    "出生地：11 北京市",
    "出生地：23 黑龙江省",
    "出生地：37 山东省",
    # "出生地：32 江苏省",
    # "出生地：41 河南省",
    # "出生地：33 浙江省",
    # "出生地：51 四川省",
}
NUM_SAMPLES = 20

SAMPLING_RATE = 16000
MAX_NEW_TOKENS = 100
NUM_BEAMS = 1
DEVICE = "cuda"

TEST_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

### 抽取处理数据

In [15]:
# ============================================================
# 2. 中文文本归一化
# ============================================================

def clean_text(text):
    """标准答案和模型输出都使用完全相同的归一化规则。"""
    text = normalize_chinese(
        text,
        remove_punctuation=True,
    )

    # OpenCC t2s 不会把“妳”转换为“你”，这里单独统一
    return text.replace("妳", "你")


# ============================================================
# 3. 抽取 20 条北京测试音频并同步到目录
# ============================================================

with TEST_TSV.open(
    encoding="utf-8",
    newline="",
) as handle:
    reader = csv.DictReader(handle, delimiter="\t")

    test_rows = []
    selected_speakers = set()

    for row in reader:
        # 必须是 not in，不能再用 != TARGET_ACCENT
        if row["accents"] not in ACCENTS:
            continue

        if row["client_id"] in selected_speakers:
            continue

        test_rows.append(row)
        selected_speakers.add(row["client_id"])

        if len(test_rows) >= NUM_SAMPLES:
            break

if len(test_rows) != NUM_SAMPLES:
    raise RuntimeError(
        f"只找到 {len(test_rows)} 条目标地区测试数据，"
        f"不足 {NUM_SAMPLES} 条"
    )

# 同步音频：从原始 clips 复制到 test1 目录
manifest_rows = []

for index, row in enumerate(test_rows, start=1):
    filename = row["path"]
    source = CV_ROOT / "clips" / filename
    destination = TEST_AUDIO_DIR / filename

    # 复制音频；重复运行时覆盖成同一份文件
    shutil.copy2(source, destination)

    manifest_rows.append({
        "序号": index,
        "文件名": filename,
        "标准答案": row["sentence"],
        "地区": row["accents"],
        "说话人": row["client_id"][:8],
    })

manifest_df = pd.DataFrame(manifest_rows)

print("测试音频目录：", TEST_AUDIO_DIR)
print("已同步音频数量：", len(manifest_df))
display(manifest_df)

测试音频目录： /sigan/ai-ml/datasets/common-voice-27-beijing-test2
已同步音频数量： 20


,序号,文件名,标准答案,地区,说话人
0,1,common_voice_zh-CN_22050002.mp3,官邸旧址已被定为长春市级文物保护单位。,出生地：11 北京市,33a20c6f
1,2,common_voice_zh-CN_22219274.mp3,开尔文是温度的计量单位。,出生地：37 山东省,16f4e754
2,3,common_voice_zh-CN_22083508.mp3,那是谁？,出生地：37 山东省,00873da0
3,4,common_voice_zh-CN_33658454.mp3,撕蛙鳄的背部中线有两排平坦鳞甲。,出生地：11 北京市,8a2fc21b
4,5,common_voice_zh-CN_22171281.mp3,场线从某一个区域的往外扩散或往内聚敛可以表达散度。,出生地：37 山东省,6271012f
5,6,common_voice_zh-CN_39053520.mp3,玩家通过全触屏进行游戏操作。,出生地：37 山东省,30f75974
6,7,common_voice_zh-CN_22302518.mp3,政府开始动员警察。,出生地：37 山东省,050fb423
7,8,common_voice_zh-CN_18757802.mp3,但外聘委员不适用委任出席规定。,出生地：11 北京市,e1b09202
8,9,common_voice_zh-CN_22187751.mp3,巴塔克所有支族都是数个家庭共同居住在一个家屋中。,出生地：37 山东省,d0d0ac81
9,10,common_voice_zh-CN_22262413.mp3,伟人后代中还有称为亚纳人的巨人。,出生地：37 山东省,d0307e61


### 加载 Processor

In [16]:
import re
import unicodedata
import opencc

_t2s = opencc.OpenCC("t2s")

def normalize_chinese(text, remove_punctuation=False):
    if text is None:
        return ""

    text = _t2s.convert(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"\s+", "", text)

    if remove_punctuation:
        text = re.sub(
            r"[，。？！：；、（）【】｛｝,\.\?!:;()\[\]{}]",
            "",
            text,
        )

    return text.strip()


def clean_text(text):
    text = normalize_chinese(
        text,
        remove_punctuation=True,
    )
    return text.replace("妳", "你")

In [17]:
# ============================================================
# 4. 加载 Processor
# ============================================================

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="Chinese",
    task="transcribe",
)

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="chinese",
    task="transcribe",
)


# ============================================================
# 5. 从同步后的目录读取音频并提取特征
# ============================================================

filenames = manifest_df["文件名"].tolist()
references = []
input_features = []

for index, row in enumerate(test_rows, start=1):
    # 注意：这里从 test1 目录读取，不再读取原始 clips
    audio_path = TEST_AUDIO_DIR / row["path"]

    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLING_RATE,
    )

    feature = processor(
        audio,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
    )["input_features"][0]

    # 标准答案也执行归一化
    references.append(
        clean_text(row["sentence"])
    )
    input_features.append(feature)

    print(f"已加载：{index}/{NUM_SAMPLES}")

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


已加载：1/20
已加载：2/20
已加载：3/20
已加载：4/20
已加载：5/20
已加载：6/20
已加载：7/20
已加载：8/20
已加载：9/20
已加载：10/20
已加载：11/20
已加载：12/20
已加载：13/20
已加载：14/20
已加载：15/20
已加载：16/20
已加载：17/20
已加载：18/20
已加载：19/20
已加载：20/20


### 生成数据

In [18]:
def run_test(use_lora):
    """
    use_lora=False：原始 Whisper large-v3
    use_lora=True ：Whisper large-v3 + LoRA
    """

    print("\n加载模型，use_lora =", use_lora)

    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_8bit=True
        ),
        device_map="auto",
    )

    # 推理阶段开启 KV Cache，减少生成耗时
    base_model.config.use_cache = True

    model = base_model

    if use_lora:
        model = PeftModel.from_pretrained(
            base_model,
            ADAPTER_DIR,
        )

    model.eval()

    # 强制使用中文转写任务
    model.generation_config.forced_decoder_ids = (
        forced_decoder_ids
    )

    predictions = []
    start_time = time.perf_counter()

    for index, feature in enumerate(
        input_features,
        start=1,
    ):
        inputs = {
            "input_features": (
                feature
                .unsqueeze(0)
                .to(DEVICE)
                .half()
            )
        }

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )

        prediction = processor.batch_decode(
            output_ids,
            skip_special_tokens=True,
        )[0]

        # 模型输出也做归一化
        predictions.append(
            clean_text(prediction)
        )

        print(f"已生成：{index}/{NUM_SAMPLES}")

    elapsed = time.perf_counter() - start_time

    # 释放显存，准备加载下一个模型
    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, elapsed


# 原始 V3
base_predictions, base_time = run_test(
    use_lora=False
)

# V3 + 北京 LoRA
lora_predictions, lora_time = run_test(
    use_lora=True
)


加载模型，use_lora = False


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


已生成：1/20
已生成：2/20
已生成：3/20
已生成：4/20
已生成：5/20
已生成：6/20
已生成：7/20
已生成：8/20
已生成：9/20
已生成：10/20
已生成：11/20
已生成：12/20
已生成：13/20
已生成：14/20
已生成：15/20
已生成：16/20
已生成：17/20
已生成：18/20
已生成：19/20
已生成：20/20

加载模型，use_lora = True
已生成：1/20
已生成：2/20
已生成：3/20
已生成：4/20
已生成：5/20
已生成：6/20
已生成：7/20
已生成：8/20
已生成：9/20
已生成：10/20
已生成：11/20
已生成：12/20
已生成：13/20
已生成：14/20
已生成：15/20
已生成：16/20
已生成：17/20
已生成：18/20
已生成：19/20
已生成：20/20


### 计算统计

In [19]:
# ============================================================
# 7. 计算总体 CER
# ============================================================

base_cer = jiwer.cer(
    reference=references,
    hypothesis=base_predictions,
)

lora_cer = jiwer.cer(
    reference=references,
    hypothesis=lora_predictions,
)

print("\n" + "=" * 50)
print("原始 V3 CER：", base_cer)
print("V3 LoRA CER：", lora_cer)
print("CER 降低量：", base_cer - lora_cer)
print("原始 V3 耗时：", round(base_time, 2), "秒")
print("LoRA 耗时：", round(lora_time, 2), "秒")
print("=" * 50)


# ============================================================
# 8. 生成逐条 results_df
# ============================================================

rows = []

for index, (
    filename,
    reference,
    base_prediction,
    lora_prediction,
) in enumerate(
    zip(
        filenames,
        references,
        base_predictions,
        lora_predictions,
    ),
    start=1,
):
    base_sample_cer = jiwer.cer(
        reference=reference,
        hypothesis=base_prediction,
    )

    lora_sample_cer = jiwer.cer(
        reference=reference,
        hypothesis=lora_prediction,
    )

    rows.append({
        "序号": index,
        "文件名": filename,
        "标准答案": reference,
        "原始V3输出": base_prediction,
        "LoRA输出": lora_prediction,
        "原始V3 CER": round(base_sample_cer, 4),
        "LoRA CER": round(lora_sample_cer, 4),
        # 正数表示 LoRA 更好
        "LoRA改善": round(
            base_sample_cer - lora_sample_cer,
            4,
        ),
    })

results_df = pd.DataFrame(rows)

display(results_df)


原始 V3 CER： 0.07744107744107744
V3 LoRA CER： 0.08417508417508418
CER 降低量： -0.006734006734006745
原始 V3 耗时： 41.21 秒
LoRA 耗时： 47.29 秒


,序号,文件名,标准答案,原始V3输出,LoRA输出,原始V3 CER,LoRA CER,LoRA改善
0,1,common_voice_zh-CN_22050002.mp3,官邸旧址已被定为长春市级文物保护单位,官邸就职已被定为长春市级文物保护单位,官邸就职已被定为长春市级文物保护单位,0.1111,0.1111,0.0000
1,2,common_voice_zh-CN_22219274.mp3,开尔文是温度的计量单位,凯尔文是温度的计量单位,凯尔文是温度的计量单位,0.0909,0.0909,0.0000
2,3,common_voice_zh-CN_22083508.mp3,那是谁,那是谁,那是谁,0.0000,0.0000,0.0000
3,4,common_voice_zh-CN_33658454.mp3,撕蛙鳄的背部中线有两排平坦鳞甲,斯瓦厄的背部中线有两排平坦鳞甲,斯瓦厄的背部中线有两排平坦鳞甲,0.2000,0.2000,0.0000
4,5,common_voice_zh-CN_22171281.mp3,场线从某一个区域的往外扩散或往内聚敛可以表达散度,场线从某一个区域的往外扩散或往内剧烈可以表达散度,场线从某一个区域的往外扩散或往内剧烈可以表达散度,0.0833,0.0833,0.0000
5,6,common_voice_zh-CN_39053520.mp3,玩家通过全触屏进行游戏操作,玩家通过全触屏进行游戏操作,玩家通过全触屏进行游戏操作,0.0000,0.0000,0.0000
6,7,common_voice_zh-CN_22302518.mp3,政府开始动员警察,政府开始动员警察,政府开始动员警察,0.0000,0.0000,0.0000
7,8,common_voice_zh-CN_18757802.mp3,但外聘委员不适用委任出席规定,但外聘委员不是用委任出席规定,但外聘人员不适用委任出席规定,0.0714,0.0714,0.0000
8,9,common_voice_zh-CN_22187751.mp3,巴塔克所有支族都是数个家庭共同居住在一个家屋中,巴塔克所有织组都是数个家庭共同居住在一个家屋中,巴塔克所有织组都是数个家庭共同居住在一个家屋中,0.0870,0.0870,0.0000
9,10,common_voice_zh-CN_22262413.mp3,伟人后代中还有称为亚纳人的巨人,伟人后代中还有称为亚纳人的巨人,伟人后代中还有称为亚纳人的巨人,0.0000,0.0000,0.0000


### 单独查看 LoRA 变差的样本

In [20]:
worse_df = results_df[
    results_df["LoRA改善"] < 0
].sort_values(
    by="LoRA改善",
    ascending=True,
)

print("\nLoRA 更差的样本：", len(worse_df))
display(worse_df)


LoRA 更差的样本： 1


,序号,文件名,标准答案,原始V3输出,LoRA输出,原始V3 CER,LoRA CER,LoRA改善
15,16,common_voice_zh-CN_22052722.mp3,毛里塔尼亚足球协会是专门管辖毛里塔尼亚足球事务的组织,毛里塔尼亚足球协会是专门管辖毛里塔尼亚足球事务的组织,毛利塔尼亚足球协会是专门管辖毛利塔尼亚足球事务的组织,0.0,0.0769,-0.0769


### 结果判断

原始 V3 CER： 0.07744107744107744

V3 LoRA CER： 0.08417508417508418

CER 降低量： -0.006734006734006745

原始 V3 耗时： 41.21 秒

LoRA 耗时： 47.29 秒


1. 数据太少了，不足以支撑训练
2. Common Voice 的 accents 表示说话人自报或出生地区，不是“这段音频确实是北京话”的标签。文本也是标准书面中文朗读，所以最终没有形成方言训练效果